# E1 — Primo modello senza training

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emanbuc/corso_foundation_tabular/blob/main/notebooks/usare-tfm/E1_primo_modello.ipynb)

**Corso 1 · Usare un TFM — Modulo 2 (laboratorio, 10 min)**

Si classifica un dataset con un tabular foundation model pre-addestrato, senza addestrare nulla e senza scegliere iperparametri.

**Idea da far emergere:** `fit` è quasi istantaneo, il lavoro avviene in `predict`.

- **Dati:** `load_breast_cancer` di scikit-learn (569 righe × 30 colonne, 2 classi).
- **Modello:** TabICLv2, tramite il pacchetto `tabicl`.

Il notebook gira anche su CPU, più lentamente.

## 0. Setup

La versione di `tabicl` è fissata: le API cambiano spesso.

Prima di proseguire: Runtime → Cambia tipo di runtime → GPU T4.

In [ ]:
%pip install -q tabicl==2.2.0

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

print("GPU disponibile:", torch.cuda.is_available())

## 1. I dati

Uno split stratificato: 75% delle righe come contesto, 25% come test.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)
print("Contesto:", X_tr.shape, " Test:", X_te.shape)

## 2. Il primo modello

L'interfaccia è quella di scikit-learn, ma `fit` e `predict` fanno cose diverse dal solito:

- `fit` memorizza il training set come **contesto**. Non aggiorna nessun peso.
- `predict_proba` esegue il transformer pre-addestrato su contesto e righe di test insieme. L'in-context learning avviene qui.

Alla prima esecuzione `fit` scarica i pesi del modello: il tempo di questa cella non fa testo.

In [ ]:
from sklearn.metrics import roc_auc_score
from tabicl import TabICLClassifier

clf = TabICLClassifier()          # nessun iperparametro da scegliere
clf.fit(X_tr, y_tr)               # memorizza il contesto, scarica i pesi al primo uso
proba = clf.predict_proba(X_te)   # l'in-context learning avviene qui
print(f"ROC-AUC: {roc_auc_score(y_te, proba[:, 1]):.4f}")

## 3. Dove va il tempo?

Ora che i pesi sono in cache, si misurano separatamente `fit` e `predict_proba`, per il TFM e per tre modelli classici con i parametri di default.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def cronometra(modello):
    """Tempo di fit, tempo di predict_proba e ROC-AUC sul test."""
    t0 = time.perf_counter()
    modello.fit(X_tr, y_tr)
    t1 = time.perf_counter()
    proba = modello.predict_proba(X_te)
    t2 = time.perf_counter()
    return {"fit (s)": t1 - t0, "predict (s)": t2 - t1, "ROC-AUC": roc_auc_score(y_te, proba[:, 1])}


modelli = {
    "TFM (TabICLv2)": TabICLClassifier(),
    "Regressione logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(random_state=0),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
risultati = pd.DataFrame({nome: cronometra(m) for nome, m in modelli.items()}).T
risultati.round(4)

**Grafico da commentare:** per ogni modello, quanto tempo va in `fit` e quanto in `predict`. L'asse è logaritmico.

In [ ]:
ax = risultati[["fit (s)", "predict (s)"]].plot.bar(figsize=(7, 3.5), logy=True, rot=15)
ax.set_ylabel("tempo (s, scala logaritmica)")
ax.set_title("Dove va il tempo: fit contro predict")
plt.tight_layout()
plt.show()

## Da osservare

- Nei modelli classici il tempo sta in `fit`: è lì che si impara dai dati. Nel TFM sta in `predict`.
- Il TFM non ha richiesto nessuna scelta: né iperparametri, né normalizzazione delle colonne.
- Su un dataset così piccolo e facile tutti i modelli hanno ROC-AUC molto alta: questo notebook mostra *come* si usa un TFM, non che sia migliore. Il confronto serio è in E4.

Domande per la discussione:

- Se `fit` non addestra nulla, da dove viene la capacità del modello di classificare questi dati?
- Cosa succede al tempo di `predict` se le righe di contesto raddoppiano? (Si riprende in E5.)
- In quale situazione un `predict` lento è un problema, e in quale no?